# Week 1 Day 2: lower-cost agent models

I compare two lower-cost candidates with the lab's GPT-6 Luna baseline. Each model gets the same claim, policy record, policy wording and scoring rule. To avoid the lab agent's multiple serial tool round trips, this notebook preloads each policy and forces one LangChain decision-tool call per claim. The first test uses three claims per model; it is not the full benchmark.

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import RateLimitError

ROOT = Path.cwd()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError("Run this notebook from the repository or one of its subfolders.")

load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise RuntimeError("Set OPENROUTER_API_KEY in the repository .env file.")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
rate_limiter = InMemoryRateLimiter(requests_per_second=0.25, max_bucket_size=1)

DATA = ROOT / "data"
claims = json.loads((DATA / "claims.json").read_text(encoding="utf-8"))
decisions = json.loads((DATA / "evals" / "decisions.json").read_text(encoding="utf-8"))
labels = {decision["claim_id"]: decision for decision in decisions}
policies = {policy["policy_id"]: policy for policy in json.loads((DATA / "policies.json").read_text(encoding="utf-8"))}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"Scoring {len(claims)} claims")

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it because a required document is missing, the story and documents disagree, or the wording does not settle the question.
"""

SINGLE_CALL_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use the customer's policy record provided with the claim. Return your decision by calling the record_decision tool exactly once.

{policy_wording}
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and record your decision.

{policy_wording}
"""

recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(
    claim_id: str,
    decision: Literal["approve", "decline", "refer"],
    payout: float,
    rationale: str,
) -> str:
    """Record a decision, payout in US dollars, and its rationale."""
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


def make_claim_prompt(claim):
    """Include the claim's policy locally to avoid extra model/tool round trips."""
    policy = policies.get(claim["policy_id"])
    if policy is None:
        raise KeyError(f"No policy record found for {claim['policy_id']}")
    return f"# Customer policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}"


def make_agent_claim_prompt(claim):
    """Send the claim only; the agent looks up its policy with a tool."""
    return f"# Claim\n\n{json.dumps(claim, indent=2)}"


def is_right(answer, label):
    """Match the handler's decision and payout within one dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def wait_for_rate_limit(error, attempt):
    """Back off exponentially, honoring OpenRouter's reset time when present."""
    delay = 2 ** attempt
    body = error.body if isinstance(error.body, dict) else {}
    metadata = body.get("metadata", {})
    headers = metadata.get("headers", {}) if isinstance(metadata, dict) else {}
    reset_ms = headers.get("X-RateLimit-Reset") if isinstance(headers, dict) else None
    if reset_ms is not None:
        delay = max(delay, float(reset_ms) / 1000 - time.time() + 1)
    print(f"Rate limit reached; retrying in {delay:.0f} seconds.", flush=True)
    time.sleep(delay)


def run_single_call_candidate(model, effort, test_claims):
    """Run one forced LangChain decision-tool call per supplied claim."""
    kwargs = {"reasoning_effort": effort} if effort else {}
    llm = ChatOpenAI(
        model=model,
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        timeout=120,
        max_retries=0,
        rate_limiter=rate_limiter,
        **kwargs,
    )
    decision_model = llm.bind_tools([record_decision], tool_choice="record_decision")
    results = []
    print(f"Starting {model}: {len(test_claims)} claims", flush=True)
    for claim_number, claim in enumerate(test_claims, start=1):
        print(f"{model}: claim {claim_number}/{len(test_claims)} {claim['claim_id']}", flush=True)
        start = time.time()
        for attempt in range(5):
            try:
                response = decision_model.invoke([
                    ("system", SINGLE_CALL_SYSTEM_PROMPT),
                    ("user", make_claim_prompt(claim)),
                ])
                break
            except RateLimitError as error:
                if attempt == 4:
                    raise
                wait_for_rate_limit(error, attempt)
        tool_calls = response.tool_calls
        if len(tool_calls) != 1 or tool_calls[0]["name"] != "record_decision":
            raise RuntimeError(f"{model} returned no valid record_decision call for {claim['claim_id']}")
        answer = tool_calls[0]["args"]
        if answer["claim_id"] != claim["claim_id"]:
            raise ValueError(f"{model} returned a decision for {answer['claim_id']} instead of {claim['claim_id']}")
        label = labels[claim["claim_id"]]
        cost = response.response_metadata["token_usage"]["cost"]
        results.append({
            "model": model,
            "claim_id": claim["claim_id"],
            **answer,
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
            "cost": cost,
            "seconds": round(time.time() - start, 1),
        })
        print(f"{model}: completed {claim['claim_id']}", flush=True)

    return results


def run_agent_candidate(model, effort, test_claims):
    """Run the assignment's two-tool LangChain agent on the supplied claims."""
    kwargs = {"reasoning_effort": effort} if effort else {}
    llm = ChatOpenAI(
        model=model,
        base_url=OPENROUTER_BASE_URL,
        api_key=openrouter_api_key,
        timeout=120,
        max_retries=0,
        rate_limiter=rate_limiter,
        **kwargs,
    )
    agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

    recorded.clear()
    results = []
    print(f"Starting {model} agent run: {len(test_claims)} claims", flush=True)
    for claim_number, claim in enumerate(test_claims, start=1):
        print(f"{model} agent: claim {claim_number}/{len(test_claims)} {claim['claim_id']}", flush=True)
        start = time.time()
        for attempt in range(5):
            try:
                result = agent.invoke({"messages": [{"role": "user", "content": make_agent_claim_prompt(claim)}]})
                break
            except RateLimitError as error:
                if attempt == 4:
                    raise
                wait_for_rate_limit(error, attempt)
        answer = recorded.get(claim["claim_id"])
        if answer is None:
            raise RuntimeError(f"{model} did not record a decision for {claim['claim_id']}")
        label = labels[claim["claim_id"]]
        cost = sum(
            message.response_metadata["token_usage"]["cost"]
            for message in result["messages"]
            if message.type == "ai"
        )
        results.append({
            "model": model,
            "claim_id": claim["claim_id"],
            **answer,
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
            "cost": cost,
            "seconds": round(time.time() - start, 1),
        })
        print(f"{model} agent: completed {claim['claim_id']}", flush=True)

    return results

In [ ]:
models = [
    ("openai/gpt-6-luna", "low"),
    ("deepseek/deepseek-v4-flash-0731", "low"),
    ("inclusionai/ling-3.0-flash", None),
]

test_claims = claims[:3]
runs = {}
for model, effort in models:
    runs[model] = run_single_call_candidate(model, effort, test_claims)

summary = []
for model, results in runs.items():
    summary.append({
        "model": model,
        "right / 3": f"{sum(result['right'] for result in results)} / {len(results)}",
        "cost": f"${sum(result['cost'] for result in results):.4f}",
        "seconds / claim": round(sum(result['seconds'] for result in results) / len(results), 1),
    })

display(pd.DataFrame(summary))

## Preliminary smoke test results

These results are from the faster single-call experiment, not the assignment benchmark.

| Model | Correct / 3 | Cost | Seconds / claim |
| --- | ---: | ---: | ---: |
| GPT-6 Luna | 3 / 3 | $0.0009 | 5.5 |
| DeepSeek V4 Flash | 3 / 3 | $0.0020 | 5.7 |
| Ling 3.0 Flash | 2 / 3 | $0.0005 | 14.1 |

## Assignment benchmark results: two-tool LangChain agent

This is one pass over all 25 claims per model, using the original `create_agent` implementation with both tools. Each model reached at least 15 / 25 in this pass, but repeat the baseline and qualifying candidates before calling any model a consistent winner.

| Model | Correct / 25 | Cost | Seconds / claim |
| --- | ---: | ---: | ---: |
| GPT-6 Luna | 17 / 25 | $0.0094 | 12.6 |
| DeepSeek V4 Flash | 16 / 25 | $0.0188 | 17.1 |
| Ling 3.0 Flash | 15 / 25 | $0.0130 | 22.2 |

In [ ]:
if len(claims) != 25:
    raise ValueError(f"Expected 25 claims, found {len(claims)}")

agent_models = [
    ("openai/gpt-6-luna", "low"),
    ("deepseek/deepseek-v4-flash-0731", "low"),
    ("inclusionai/ling-3.0-flash", None),
]

agent_runs = {}
for model, effort in agent_models:
    agent_runs[model] = run_agent_candidate(model, effort, claims)

agent_summary = []
for model, results in agent_runs.items():
    agent_summary.append({
        "model": model,
        "right / 25": f"{sum(result['right'] for result in results)} / {len(results)}",
        "cost": f"${sum(result['cost'] for result in results):.4f}",
        "seconds / claim": round(sum(result['seconds'] for result in results) / len(results), 1),
    })

display(pd.DataFrame(agent_summary))